# Titanic ③ グラフで傾向を確かめる

01 と 02 では表の数字で傾向を見た．ここでは同じデータをグラフにして，次に作る特徴量の手がかりを探す．

グラフを描く関数は `src/titanic/visualize.py` にまとめてある．新しい特徴量を作ったときも，同じ関数ですぐに確かめられる．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from titanic.data import load_data
from titanic.visualize import plot_hist_by_survival, plot_survival_heatmap, plot_survival_rate, set_style

set_style()
train, _ = load_data()

## カテゴリごとの生存率

棒の下の n は，その値の人数．人数が少ない値の生存率はたまたまの影響を受けやすいので，当てにしすぎないようにする．

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, column in zip(axes, ["Sex", "Pclass", "Embarked"]):
    plot_survival_rate(train, column, ax=ax)
plt.show()

性別と等級は 01 で見たとおり．乗船港は C（Cherbourg）だけ生存率が高い．乗った港そのものが生死を分けたとは考えにくいので，港ごとの等級を見てみる．

In [ ]:
plot_survival_heatmap(train, "Embarked", "Pclass")
plt.show()

# 港ごとの等級の割合（行ごとに足すと 1 になる）
print(pd.crosstab(train["Embarked"], train["Pclass"], normalize="index").round(2))

C から乗った人の半分は 1 等で，S（2 割）や Q（ほぼいない）と比べて 1 等の割合が大きい．C の生存率が高い理由の多くは，この等級の構成で説明できそうだ．

ただ，同じ 3 等どうしでも C は 38%，S は 19% と差があり，等級だけでは説明しきれない．港によって男女比が違うことも絡んでいるのかもしれない．

ある列が効いているように見えて，実は別の列の影響だった，ということ（交絡）はよくある．気になる差が出たら，ほかの列との組み合わせで確かめるようにしたい．

## 家族の人数

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, column in zip(axes, ["SibSp", "Parch"]):
    plot_survival_rate(train, column, ax=ax)
plt.show()

SibSp も Parch も，0（一人）より 1〜2 のほうが生存率が高く，多すぎると下がる．どちらも「一緒に乗っていた家族」を別々に数えているだけなので，足して家族の人数にまとめると傾向がはっきりするかもしれない．

In [ ]:
# 家族の人数 = 兄弟姉妹・配偶者 + 親・子 + 本人
family_size = train["SibSp"] + train["Parch"] + 1
fig, ax = plt.subplots(figsize=(8, 3.5))
plot_survival_rate(train.assign(FamilySize=family_size), "FamilySize", ax=ax)
plt.show()

一人だと 30%，2〜4 人だと 55〜72% で，5 人以上になると低い．山なりの関係なので，ロジスティック回帰のような線形モデルにはそのままだと扱いにくい．「一人」「2〜4 人」「5 人以上」のように区切った特徴量にすると使いやすいと思う．

## 性別 × 等級

In [ ]:
plot_survival_heatmap(train, "Sex", "Pclass")
plt.show()

02 でランダムフォレストが拾った「3 等の女性は半々」という差が，ひと目で分かる．1 等の男性（37%）も，2・3 等の男性（16%・14%）よりかなり高い．

## 年齢と運賃の分布

運賃は 500 を超える人が 3 人だけいて，そのまま描くとほかが潰れてしまう．そこで 0〜300 の範囲だけ描く．

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
plot_hist_by_survival(train, "Age", bins=np.arange(0, 85, 5), ax=axes[0])
plot_hist_by_survival(train, "Fare", bins=np.arange(0, 301, 10), ax=axes[1])
plt.show()

年齢では，5 歳未満は生存のほうがはっきり多く，15〜35 歳は死亡が大きく上回る．運賃は 10 未満の安い切符に死亡が集中していて，50 を超えたあたりから生存のほうが多くなる．運賃は等級とほぼ連動しているので，等級と同じ情報をなぞっている部分も大きいはず．

## 年齢帯ごとの生存率

ヒストグラムは人数を数えたものなので，生存率は年齢を区間に分けて見る．`pd.cut` で作った区間は `(0, 10]` のように表示され，「0 より大きく 10 以下」という意味になる．

In [ ]:
age_band = pd.cut(train["Age"], bins=[0, 10, 20, 30, 40, 50, 60, 80])
fig, ax = plt.subplots(figsize=(8, 3.5))
plot_survival_rate(train.assign(AgeBand=age_band), "AgeBand", ax=ax)
plt.show()

10 歳以下は 59% と高く，60 歳を超えると 23% まで下がる．その間は 37〜45% でほぼ横ばいだった．年齢が効いているのは，主に「子どもかどうか」の部分なのだと考えている．Age は 177 人分欠けているので，名前の敬称（Master など）から子どもかどうかを補えると良さそうだ．

## まとめ: 次に試したい特徴量

- **家族の人数**（SibSp + Parch + 1）．一人，2〜4 人，5 人以上で生存率がはっきり違う
- **名前の敬称**（Mr / Mrs / Miss / Master）．性別と「子どもかどうか」を一度に表せて，Age の欠損も補える
- **乗船港**は等級と絡んでいるので，ほかの特徴量を入れたうえで効くかどうかを交差検証で確かめる